# 08 Advanced Challenge - Data QC and Outlier Visualization in Python

## Biochemistry question

In this synthetic assay dataset, which measurements should be reviewed before interpreting the overall pattern?


In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

df = pd.read_csv("../data/qc_outlier_sample.csv")
df.head()

In [ ]:
# Compute z-score within each drug/concentration group.
df["group_key"] = df["drug_name"] + "_" + df["concentration_uM"].astype(str)

df["group_mean"] = df.groupby("group_key")["cell_viability_percent"].transform("mean")
df["group_sd"] = df.groupby("group_key")["cell_viability_percent"].transform("std")
df["z_score"] = (df["cell_viability_percent"] - df["group_mean"]) / df["group_sd"]

df["qc_flag"] = np.where(df["z_score"].abs() > 1.5, "review", "ok")
df

In [ ]:
fig = px.scatter(
    df,
    x="concentration_uM",
    y="cell_viability_percent",
    color="qc_flag",
    symbol="drug_name",
    hover_data=["sample_id", "drug_name", "replicate", "z_score"],
    title="QC Scatter Plot: Possible Outliers"
)
fig.update_xaxes(type="log")
# If this chart does not render in Jupyter, try: fig.show(renderer="iframe") or fig.show(renderer="browser")
fig.show()


In [ ]:
fig2 = px.box(
    df,
    x="drug_name",
    y="cell_viability_percent",
    color="drug_name",
    points="all",
    title="Cell Viability Distribution by Drug"
)
# If this chart does not render in Jupyter, try: fig2.show(renderer="iframe") or fig2.show(renderer="browser")
fig2.show()


## Interpretation Questions

1. Which points were flagged for review?
2. Are flagged points always wrong?
3. What should a lab scientist check before removing an outlier?
4. How could this QC view support a future educational BioDose workflow?

## Limitations

- This is synthetic assay data for learning QC ideas.
- A z-score flag is a review prompt, not proof that a point is invalid.
- Real QC decisions require raw data, lab notes, instrument context, and study design details.
